# R07-H34 - seed attribution: does the vector index find the entry node?

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-06 <br>
**Pipeline stage**: R07 contrarian slate, hypothesis 1 of 16 <br>
**Graph**: rebuilt CPAP corpus (neo4j2), Titan embeddings, zero completions <br>

The vector index is the ONLY entry into the graph (`_retrieve_local`: question embedding -> top-k seeds -> PPR/propositions/aliases expand from there). P09 proved a gold node with near-zero question similarity is invisible to it. H34 measures, per gold-evidence string, WHICH channel actually surfaced it - and ablates each channel to quantify what pure vector seeding delivers alone.

Attribution channels, checked in production order (first hit wins):
- **vec** - the gold string appears in a direct vector-seed node's own render (properties, description, relations)
- **alias** - only in a seed's SAME_AS alias-cluster contribution (H21 machinery)
- **prop_text** - only in a retrieved proposition's text (R02-H11 fact lines)
- **prop_node** - only in the render of a proposition-seeded node (R03-H14)
- **ppr** - only in the render of a PPR-expansion-only node (R01-H2)

## Approach
1. **Harvest** - per probe: one Titan embedding, then each retrieval channel run separately with production parameters
2. **Attribute** - each gold string classified to the first channel containing it (normalized substring, the H22 scorer convention)
3. **Ablate** - evidence recall of pure-vector context vs the full production `_retrieve_local` context
4. **Verdict** - against the pre-registered bar: confirmed if direct-seed share <= 60% OR pure-vector ablation drops recall >= 25%; refuted if pure vector alone reaches >= 90% of full-pipeline recall

## Outputs
- `reports/probe-eval-r07h34-<stamp>.json` - per-gold attribution rows, channel census, ablation recalls, verdict
- In-notebook: attribution table, ablation summary, verdict

In [1]:
# Imports
# stdlib
import datetime  # report stamps
import json  # report persistence
import os  # graph selection env
import re  # evidence normalization

# third party
import yaml  # probe set
from pathlib import Path
from rich import print as rprint  # semantic output
from rich.progress import Progress  # harvest loop

os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j2:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

# project
from knowledge_graph_foundry import Foundry, load_settings  # pipeline
from knowledge_graph_foundry.extraction import generate_embeddings  # Titan probe embedding
from knowledge_graph_foundry.graph.graphrag import ppr_query, vector_query  # retrieval channels
from knowledge_graph_foundry.graph.propositions import proposition_query  # proposition channel
from knowledge_graph_foundry.models import Entity  # embedding probe wrapper

2026-07-06 19:08:16.903 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# Configuration
PROBES_PATH = Path("../tests/probes/cpap-probe-set.yml")  # 28 probes with gold evidence

settings = load_settings(Path("../config.yml"))
settings.graphrag.propositions_enabled = True   # production channel state
settings.graphrag.abstention_enabled = False    # measurement, not answering

TOP_K = settings.graphrag.top_k                          # vector seeds per probe
PPR_TOP_N = settings.graphrag.ppr_top_n                  # PPR nodes into context
PPR_DAMPING = settings.graphrag.ppr_damping              # PageRank damping
PROP_TOP_K = settings.graphrag.proposition_top_k         # propositions per probe
VEC_INDEX = settings.graphrag.vector_index_name          # entity vector index
PROP_INDEX = settings.graphrag.proposition_index_name    # proposition vector index

# pre-registered acceptance bar (experiments log, R07-H34)
BAR_DIRECT_SHARE = 0.60   # confirmed if direct-seed share <= this
BAR_ABLATION_DROP = 0.25  # ... OR pure-vector recall drop >= this
BAR_REFUTE_RATIO = 0.90   # refuted if pure-vector >= this fraction of full recall

probes = yaml.safe_load(PROBES_PATH.read_text())
gold_probes = [p for p in probes if p.get("gold_evidence")]

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"\u2500" * 40}[/dim]
[bold]Graph[/bold]
  Neo4j: [cyan]{os.environ['NEO4J_URI']}[/cyan]
  Vector index: [cyan]{VEC_INDEX}[/cyan]  Proposition index: [cyan]{PROP_INDEX}[/cyan]

[bold]Channels (production parameters)[/bold]
  Vector top-k: [yellow]{TOP_K}[/yellow]
  PPR top-n: [yellow]{PPR_TOP_N}[/yellow] [dim](damping {PPR_DAMPING})[/dim]
  Proposition top-k: [yellow]{PROP_TOP_K}[/yellow]

[bold]Probes[/bold]
  Set: [cyan]{PROBES_PATH}[/cyan] ([yellow]{len(probes)}[/yellow] total, [yellow]{len(gold_probes)}[/yellow] with gold evidence)

[bold]Acceptance bar[/bold]
  Confirmed: direct-seed share <= [yellow]{BAR_DIRECT_SHARE}[/yellow] OR ablation drop >= [yellow]{BAR_ABLATION_DROP}[/yellow]
  Refuted: pure-vector recall >= [yellow]{BAR_REFUTE_RATIO}[/yellow] x full recall
""")


def _norm(s):
    return re.sub(r"\s+", " ", s.casefold())

Configuration
────────────────────────────────────────
Graph
  Neo4j: bolt://user-konrad.jelen-kgf-neo4j2:7687
  Vector index: kgf_entity_embeddings  Proposition index: kgf_proposition_embeddings

Channels (production parameters)
  Vector top-k: 8
  PPR top-n: 15 (damping 0.85)
  Proposition top-k: 8

Probes
  Set: ../tests/probes/cpap-probe-set.yml (28 total, 24 with gold evidence)

Acceptance bar
  Confirmed: direct-seed share <= 0.6 OR ablation drop >= 0.25
  Refuted: pure-vector recall >= 0.9 x full recall

## Channel harvest

Per probe: one Titan embedding, then each channel run separately with production parameters. `render_nodes` replicates the production node render (`_retrieve_local` block build) with the alias-cluster merge switchable, so alias contributions are separable from a seed's own facts. The full-pipeline baseline uses `Foundry._retrieve_local` itself - the exact context a reader would see.

In [3]:
def render_nodes(session, node_ids, include_alias=True):
    """Replicates the production per-node render: description + prop_* spec +
    currently-valid relations, plus (optionally) the R04-H21 alias-cluster
    property merge and 'Also known as' line."""
    blocks = []
    for nid in node_ids:
        row = session.run(
            "MATCH (e:Entity {id: $id}) RETURN e.name AS name, labels(e) AS types, "
            "e.description AS description, properties(e) AS props",
            id=nid,
        ).single()
        if row is None:
            continue
        spec = {k.removeprefix("prop_"): v for k, v in row["props"].items() if k.startswith("prop_")}
        alias_names = []
        if include_alias:
            aliases = session.run(
                "MATCH (e:Entity {id: $id})-[:SAME_AS*1..2]-(a:Entity) "
                "WHERE a.id <> $id RETURN DISTINCT a.name AS name, properties(a) AS props LIMIT 5",
                id=nid,
            ).data()
            alias_names = [a["name"] for a in aliases]
            for a in aliases:
                for k, v in a["props"].items():
                    if k.startswith("prop_"):
                        spec.setdefault(k.removeprefix("prop_"), v)
        rels = session.run(
            "MATCH (e:Entity {id: $id})-[r]-(n:Entity) "
            "WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' "
            "RETURN type(r) AS rel, n.name AS name LIMIT 15",
            id=nid,
        ).data()
        blocks.append(
            f"## {row['name']} ({', '.join(row['types'])})\n"
            + (f"Also known as: {', '.join(alias_names)}\n" if alias_names else "")
            + f"{row['description'] or ''}\n"
            f"Properties: {json.dumps(spec, default=str)}\n"
            "Relations: " + "; ".join(f"{r['rel']} -> {r['name']}" for r in rels)
        )
    return "\n".join(blocks)


harvest = {}
with Foundry(settings) as f, Progress() as progress:
    task = progress.add_task("channel harvest", total=len(gold_probes))
    for p in gold_probes:
        q = p["question"]
        probe_e = Entity.create(q[:80], types=["Query"], description=q)
        emb = generate_embeddings([probe_e], settings.embeddings)[0].embedding

        seeds = vector_query(f.driver, emb, VEC_INDEX, top_k=TOP_K)
        seed_ids = [s["id"] for s in seeds]
        prop_hits = proposition_query(f.driver, emb, PROP_INDEX, top_k=PROP_TOP_K)
        prop_seed_ids = [
            eid for h in prop_hits for eid in h["entity_ids"] if eid not in seed_ids
        ]
        ranked = ppr_query(f.driver, seed_ids + prop_seed_ids, top_n=PPR_TOP_N, damping=PPR_DAMPING)
        known = set(seed_ids) | set(prop_seed_ids)
        ppr_only_ids = [n["id"] for n in ranked if n["id"] not in known]

        with f.driver.session() as session:
            ctx_vec = render_nodes(session, seed_ids, include_alias=False)
            ctx_vec_alias = render_nodes(session, seed_ids, include_alias=True)
            ctx_prop_nodes = render_nodes(session, prop_seed_ids, include_alias=True)
            ctx_ppr = render_nodes(session, ppr_only_ids, include_alias=True)
        ctx_prop_text = "\n".join(h["text"] for h in prop_hits)
        full_lines, _, _ = f._retrieve_local(q)

        harvest[p["id"]] = {
            "question": q,
            "gold": p["gold_evidence"],
            "n_seeds": len(seed_ids),
            "n_prop_seeds": len(prop_seed_ids),
            "n_ppr_only": len(ppr_only_ids),
            "ctx": {
                "vec": ctx_vec,
                "vec_alias": ctx_vec_alias,
                "prop_text": ctx_prop_text,
                "prop_node": ctx_prop_nodes,
                "ppr": ctx_ppr,
                "full": "\n".join(full_lines),
            },
        }
        progress.advance(task)

rprint(f"[green]harvested[/green] {len(harvest)} probes x 6 contexts")

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-07-06 19:08:17.655 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 
1/1

2026-07-06 19:08:17.657 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure 
field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition 
line=1, column=1, offset=0>, raw_classification='DEPRECATION', 
classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', 
'_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0',
'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature 
deprecated without replacement. id is deprecated and will be removed without a replacement.', 
position=<SummaryInputPosition line=1, column=44, offset=43>, raw_classification='DEPRECATION', 
classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', 
'_severity': 'WARNING', '_position': {'offset': 43, 'line': 1, 'column': 44}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (e:Entity) WHERE e.id IN $ids RETURN id(e) AS nid'

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

harvested 24 probes x 6 contexts

## Attribution, ablation and verdict

Each gold string is classified to the first channel containing it (production order: seed's own render -> alias contribution -> proposition text -> proposition-seeded node -> PPR-only node). The ablation compares evidence recall of the pure-vector context (seeds' own renders, no aliases, no propositions, no PPR) against the full production context. Verdict against the pre-registered bar.

In [4]:
CHANNEL_ORDER = ["vec", "alias", "prop_text", "prop_node", "ppr", "missing"]

# iteration 2 matcher: strict substring missed surface variants ("28 dB(A)"
# vs "28 dBA", "275mm" vs "275 mm") - 23/33 golds read "missing" while the
# scoreboard passes 28/28. present() adds the H22 scorer conventions: value-
# token majority for numeric golds, 0.6 content-word overlap for prose golds.
def value_tokens(text):
    return re.findall(r"[\w.\-/]*\d[\w.\-/]*", text)


# iteration 3: units can live in the property KEY (dimensions_mm: "275 x
# 170 x 140") while the gold carries them inline ("275mm x 170mm x 140mm")
# - strip unit suffixes from the gold and squash-match the numeric skeleton
_UNIT = r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"


def present(gold, ctx_norm):
    ng = _norm(gold)
    if ng in ctx_norm:
        return True
    squashed = re.sub(r"[\s,()]", "", ctx_norm)
    skeleton = re.sub(r"[\s,()]", "", re.sub(_UNIT, "", ng))
    if any(ch.isdigit() for ch in skeleton) and len(skeleton) >= 5 and skeleton in squashed:
        return True
    tokens = value_tokens(gold)
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ctx_norm or re.sub(r"[\s,()]", "", _norm(t)) in squashed)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng))
    ctx_words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx_norm))
    return bool(words) and len(words & ctx_words) / len(words) >= 0.6


rows = []
per_probe = {}
for pid, h in harvest.items():
    ctx = {k: _norm(v) for k, v in h["ctx"].items()}
    found_vec = found_full = 0
    for g in h["gold"]:
        if present(g, ctx["vec"]):
            channel = "vec"
        elif present(g, ctx["vec_alias"]):
            channel = "alias"
        elif present(g, ctx["prop_text"]):
            channel = "prop_text"
        elif present(g, ctx["prop_node"]):
            channel = "prop_node"
        elif present(g, ctx["ppr"]):
            channel = "ppr"
        else:
            channel = "missing"
        in_full = present(g, ctx["full"])
        found_vec += channel == "vec"
        found_full += in_full
        rows.append({"probe": pid, "gold": g[:70], "channel": channel, "in_full_context": in_full})
    n = len(h["gold"])
    per_probe[pid] = {"recall_vec": found_vec / n, "recall_full": found_full / n}

census = {c: sum(1 for r in rows if r["channel"] == c) for c in CHANNEL_ORDER}
found_any = sum(v for c, v in census.items() if c != "missing")
direct_share = census["vec"] / found_any if found_any else 0.0
recall_vec = sum(v["recall_vec"] for v in per_probe.values()) / len(per_probe)
recall_full = sum(v["recall_full"] for v in per_probe.values()) / len(per_probe)
drop = (recall_full - recall_vec) / recall_full if recall_full else 0.0
rescued = [
    pid for pid, v in per_probe.items() if v["recall_vec"] == 0 and v["recall_full"] > 0
]

confirmed = direct_share <= BAR_DIRECT_SHARE or drop >= BAR_ABLATION_DROP
refuted = recall_full > 0 and (recall_vec / recall_full) >= BAR_REFUTE_RATIO
verdict = "REFUTED" if refuted else ("CONFIRMED" if confirmed else "INCONCLUSIVE")

rprint(f"""[bold cyan]Seed attribution - channel census[/bold cyan]
[dim]{"\u2500" * 40}[/dim]""")
for c in CHANNEL_ORDER:
    rprint(f"  {c:>10}: [yellow]{census[c]}[/yellow]")
for r in rows:
    if r["channel"] != "vec":
        rprint(f"  [dim]{r['probe']} [{r['channel']}] {r['gold']}[/dim]")

rprint(f"""
[bold cyan]Ablation + verdict[/bold cyan]
[dim]{"\u2500" * 40}[/dim]
  Gold strings: [yellow]{len(rows)}[/yellow] ([yellow]{found_any}[/yellow] surfaced by any channel)
  Direct-seed share: [yellow]{direct_share:.2f}[/yellow] [dim](bar: <= {BAR_DIRECT_SHARE})[/dim]
  Evidence recall - pure vector: [yellow]{recall_vec:.3f}[/yellow]  full pipeline: [yellow]{recall_full:.3f}[/yellow]
  Ablation drop: [yellow]{drop:.2%}[/yellow] [dim](bar: >= {BAR_ABLATION_DROP:.0%})[/dim]
  Probes rescued entirely by non-vector channels: [yellow]{rescued}[/yellow]
  Verdict: [{'green' if verdict == 'CONFIRMED' else 'red' if verdict == 'REFUTED' else 'yellow'}]{verdict}[/]
""")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"probe-eval-r07h34-{stamp}.json"
out.write_text(json.dumps({
    "census": census, "direct_seed_share": direct_share,
    "recall_pure_vector": recall_vec, "recall_full": recall_full,
    "ablation_drop": drop, "rescued_probes": rescued,
    "per_probe": per_probe, "rows": rows, "verdict": verdict,
    "channel_sizes": {pid: {k: h[k] for k in ("n_seeds", "n_prop_seeds", "n_ppr_only")}
                      for pid, h in harvest.items()},
}, indent=2, default=str))
rprint("saved", str(out))

Seed attribution - channel census
────────────────────────────────────────

vec: 21

alias: 2

prop_text: 2

prop_node: 8

...

P18  27 dBA

P19  constant lower pressure

P22  amplitude of oscillations

Ablation + verdict
────────────────────────────────────────
  Gold strings: 33 (33 surfaced by any channel)
  Direct-seed share: 0.64 (bar: <= 0.6)
  Evidence recall - pure vector: 0.667  full pipeline: 1.000
  Ablation drop: 33.33% (bar: >= 25%)
  Probes rescued entirely by non-vector channels: ['P01', 'P03', 'P09', 'P19', 'P22']
  Verdict: CONFIRMED

saved ../reports/probe-eval-r07h34-20260706-170830.json